# Three ways to group the CWE catalog

Every file in `cwes/weakness/` is one weakness. This notebook assigns each of those weaknesses to the groups MITRE already publishes in the same catalog. Pull requests and CVEs are not used.

The three groupings answer different questions:

1. **Research Concepts (view 1000).** What kind of mistake is this? Each weakness follows its primary parent up to one pillar.
2. **Software Development (view 699).** Where would a developer look this up? These are the category pages on the MITRE site. A weakness can sit in more than one.
3. **Common consequence scopes.** What harm does the weakness cause? Taken from `CommonConsequences` on the weakness itself. A weakness usually lists more than one scope.

Categories and views are the grouping nodes, so they are not counted as weaknesses. The notebook needs `matplotlib` and `pandas`, and it expects to be run from `crest-pr-miner` (or any directory above the `cwes/` folder).


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 11,
})

TAG = re.compile(r"<[^>]+>")
ROLES = re.compile(r"\s*\(Roles:.*$")


def find_cwe_dir() -> Path:
    """Locate cwes/ from the working directory or a parent of it."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        for path in (candidate / "cwes", candidate / "crest-pr-miner" / "cwes"):
            if (path / "weakness").is_dir() and (path / "view" / "cwe-1000.json").is_file():
                return path
    raise FileNotFoundError(
        "Could not find the CWE catalog. Run this notebook from crest-pr-miner, "
        "next to the cwes/ directory."
    )


def load_entries(kind: str) -> dict[str, dict]:
    entries = {}
    for path in (CWE_DIR / kind).glob("cwe-*.json"):
        record = json.loads(path.read_text(encoding="utf-8"))
        entries[str(record["ID"])] = record
    return entries


def first_sentence(text: str | None, limit: int = 340) -> str:
    """First sentence of a catalog description, with HTML tags removed."""
    cleaned = ROLES.sub("", TAG.sub(" ", text or ""))
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return ""
    sentence = re.split(r"(?<=[.])\s+", cleaned, maxsplit=1)[0]
    if len(sentence) > limit:
        return sentence[: limit - 1].rstrip() + "…"
    return sentence


def record_name(record: dict) -> str:
    return record.get("Name") or "?"


def cwe_label(cwe_id: str) -> str:
    return f"CWE-{cwe_id}"


def unmapped_table(cwe_ids: list[str]) -> pd.DataFrame:
    """Small status table for weaknesses that did not map into a grouping."""
    return pd.DataFrame({
        "CWE": [cwe_label(cwe_id) for cwe_id in cwe_ids],
        "Name": [record_name(weaknesses[cwe_id]) for cwe_id in cwe_ids],
        "Status": [weaknesses[cwe_id].get("Status", "") for cwe_id in cwe_ids],
    }).set_index("CWE")


def bar_chart(names: list[str], counts: list[int], title: str, color: str) -> None:
    height = max(3.4, 0.38 * len(names))
    fig, ax = plt.subplots(figsize=(11, height), layout="constrained")
    y = range(len(names))

    ax.barh(y, counts, color=color, height=0.72)
    ax.set_yticks(y, names)
    ax.invert_yaxis()
    ax.set_xlabel("Weaknesses")
    ax.set_title(title)
    ax.set_xlim(0, (max(counts) if counts else 1) * 1.22)
    ax.tick_params(axis="y", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for i, count in enumerate(counts):
        ax.text(count, i, f"  {count}", va="center", ha="left", fontsize=8, color="#333333", clip_on=False)

    display(fig)
    plt.close(fig)


def show_groups(groups: list[dict], title: str, color: str) -> None:
    """Bar chart plus a table of MITRE's one-line description for each group."""
    ordered = sorted(groups, key=lambda g: (-g["count"], g["name"]))
    names = [g["name"] for g in ordered]
    counts = [g["count"] for g in ordered]
    about = [g["about"] for g in ordered]
    labels = [g.get("label", cwe_label(g["id"])) for g in ordered]

    bar_chart(names, counts, title, color)

    if labels == names:
        table = pd.DataFrame({"Scope": names, "Weaknesses": counts, "What it covers": about})
    else:
        table = pd.DataFrame({
            "CWE": labels,
            "Name": names,
            "Weaknesses": counts,
            "What it covers": about,
        }).set_index("CWE")

    with pd.option_context("display.max_rows", 80, "display.max_colwidth", 280):
        display(table)


CWE_DIR = find_cwe_dir()
weaknesses = load_entries("weakness")
categories = load_entries("category")
views = load_entries("view")

print(
    f"Catalog: {CWE_DIR}\n"
    f"{len(weaknesses):,} weaknesses, {len(categories):,} categories, {len(views):,} views."
)


## 1. Research Concepts pillars

View 1000 is MITRE's hierarchy of the kind of mistake, independent of language and of where the bug appears in a product. The roots are **pillars**. Every other weakness has one primary parent in this view: a `ChildOf` link with `ViewID` 1000 and `Ordinal` Primary. This notebook walks that parent until it reaches a pillar, so each weakness lands in exactly one pillar.

A pillar is still a weakness. It is the broadest description of that family. Class, base, and variant weaknesses underneath it are more specific versions of the same kind of mistake.


In [ ]:
VIEW_RESEARCH = "1000"
pillar_ids = [str(m["CweID"]) for m in views[VIEW_RESEARCH]["Members"]]
pillar_set = set(pillar_ids)


def primary_parent(record: dict) -> str | None:
    """Primary ChildOf parent in view 1000; fall back to any ChildOf in that view."""
    relations = record.get("RelatedWeaknesses") or []
    child_of = [
        rel for rel in relations
        if rel.get("Nature") == "ChildOf" and str(rel.get("ViewID")) == VIEW_RESEARCH
    ]
    primary = [rel for rel in child_of if rel.get("Ordinal") == "Primary"]
    chosen = primary or child_of
    return str(chosen[0]["CweID"]) if chosen else None


def pillar_of(cwe_id: str) -> str | None:
    """Walk primary parents until a Research Concepts pillar is reached."""
    seen: set[str] = set()
    current = cwe_id
    while current and current not in seen:
        seen.add(current)
        record = weaknesses.get(current)
        if record is None:
            return None
        if current in pillar_set or record.get("Abstraction") == "Pillar":
            return current
        current = primary_parent(record)
    return None


pillar_by_weakness = {cwe_id: pillar_of(cwe_id) for cwe_id in weaknesses}
unmapped_pillars = sorted(
    (cwe_id for cwe_id, pillar_id in pillar_by_weakness.items() if pillar_id is None),
    key=int,
)

pillar_groups = []
for pillar_id in pillar_ids:
    record = weaknesses[pillar_id]
    pillar_groups.append({
        "id": pillar_id,
        "name": record_name(record),
        "count": sum(1 for found in pillar_by_weakness.values() if found == pillar_id),
        "about": first_sentence(record.get("Description")),
    })

mapped = len(weaknesses) - len(unmapped_pillars)
print(
    f"{views[VIEW_RESEARCH]['Name']} (view {VIEW_RESEARCH}). "
    f"{mapped:,} of {len(weaknesses):,} weaknesses reach a pillar. "
    f"{len(unmapped_pillars):,} do not."
)
show_groups(pillar_groups, "Weaknesses by research-concept pillar", "#2A6EBB")

unmapped_status = sorted({weaknesses[cwe_id].get("Status", "") for cwe_id in unmapped_pillars})
display(Markdown(
    f"**Unmapped ({len(unmapped_pillars)}).** "
    "These entries have no parent in view 1000. "
    f"Their status values are: {', '.join(unmapped_status) or 'none'}. "
    "Deprecated entries are kept in the catalog so old identifiers still resolve, "
    "and MITRE no longer places them in the hierarchy."
))
display(unmapped_table(unmapped_pillars))


## 2. Software Development categories

View 699 is the Software Development view on the MITRE site. Its top level is a set of categories named for things developers already recognize: authentication, cryptography, memory buffers, concurrency, and so on. Membership is stored on the category, in `Relationships` with `ViewID` 699, and the same weakness is allowed to appear under more than one category.

This view is shallow on purpose. MITRE mostly lists base-level weaknesses a developer might search for, and leaves out many classes and variants. A weakness with no category here is unmapped. That is how the view is built.


In [ ]:
VIEW_SOFTWARE = "699"


def category_member_ids(category_id: str, seen: set[str] | None = None) -> set[str]:
    """Weaknesses under this category in view 699, expanding nested categories."""
    seen = set() if seen is None else set(seen)
    if category_id in seen:
        return set()
    seen.add(category_id)

    record = categories.get(category_id)
    if record is None:
        return {category_id} if category_id in weaknesses else set()

    found: set[str] = set()
    for rel in record.get("Relationships") or []:
        if str(rel.get("ViewID")) != VIEW_SOFTWARE:
            continue
        member_id = str(rel["CweID"])
        if member_id in categories:
            found |= category_member_ids(member_id, seen)
        elif member_id in weaknesses:
            found.add(member_id)
    return found


software_category_ids = [str(m["CweID"]) for m in views[VIEW_SOFTWARE]["Members"]]
members_by_category = {
    category_id: category_member_ids(category_id)
    for category_id in software_category_ids
}

categories_by_weakness: dict[str, list[str]] = {cwe_id: [] for cwe_id in weaknesses}
for category_id, member_ids in members_by_category.items():
    for member_id in member_ids:
        categories_by_weakness.setdefault(member_id, []).append(category_id)

category_groups = [{
    "id": category_id,
    "name": record_name(categories[category_id]),
    "count": len(members_by_category[category_id]),
    "about": first_sentence(categories[category_id].get("Summary")),
} for category_id in software_category_ids]

in_some_category = sum(1 for ids in categories_by_weakness.values() if ids)
in_several = sum(1 for ids in categories_by_weakness.values() if len(ids) > 1)
unmapped_categories = sorted(
    (cwe_id for cwe_id, ids in categories_by_weakness.items() if not ids),
    key=int,
)

print(
    f"{views[VIEW_SOFTWARE]['Name']} (view {VIEW_SOFTWARE}): "
    f"{len(software_category_ids)} categories.\n"
    f"{in_some_category:,} of {len(weaknesses):,} weaknesses appear in at least one category. "
    f"{in_several:,} appear in more than one, so the bars are not a partition.\n"
    f"{len(unmapped_categories):,} weaknesses are in none of these categories."
)
show_groups(category_groups, "Weaknesses by Software Development category", "#E64B35")
display(Markdown(
    f"**Not in this view ({len(unmapped_categories)}).** "
    "View 699 is a browsing aid, not a complete partition. "
    "A class such as CWE-119 (buffer errors) can be absent here while a more specific "
    "weakness under it, such as CWE-787, is listed. "
    "The research-concept pillar above is the grouping that still covers those entries."
))


## 3. Common consequence scopes

On each weakness page, Common Consequences says what goes wrong if the weakness is present. The **scope** is the security property that is harmed. MITRE draws that property from a fixed list. The sentence under each scope below is that list, in plain language. The wording of each weakness's own consequence note stays on the weakness page.

A weakness usually names several scopes, so a single weakness is counted in each bar it touches. The bars add up to more than the number of weaknesses.


In [ ]:
from collections import Counter

SCOPE_ABOUT = {
    "Confidentiality": "Information can be disclosed to an actor who is not allowed to see it.",
    "Integrity": "Data or code can be changed, or the product can be made to behave incorrectly.",
    "Availability": "A resource or service can be made unavailable or degraded.",
    "Access Control": "A control that decides who may use a resource can be bypassed or weakened.",
    "Accountability": "An action cannot be reliably traced to the actor who performed it.",
    "Authentication": "An identity claim can be accepted without a proper check of who the actor is.",
    "Authorization": "An actor can perform an action without the permission the product meant to require.",
    "Non-Repudiation": "An actor can deny having taken an action, because the product keeps no reliable proof of it.",
    "Other": "MITRE records a consequence here when the harm does not fit one of the named scopes.",
}


def scopes_of(record: dict) -> list[str]:
    """Unique consequence scopes listed on a weakness, in catalog order."""
    found: list[str] = []
    for consequence in record.get("CommonConsequences") or []:
        raw = consequence.get("Scope") or []
        if isinstance(raw, str):
            raw = [raw]
        for scope in raw:
            if scope not in found:
                found.append(scope)
    return found


scopes_by_weakness = {cwe_id: scopes_of(record) for cwe_id, record in weaknesses.items()}
scope_counts = Counter(
    scope for scope_names in scopes_by_weakness.values() for scope in scope_names
)

scope_groups = [{
    "id": scope,
    "label": scope,
    "name": scope,
    "count": count,
    "about": SCOPE_ABOUT.get(scope, "A scope listed by MITRE that is outside the usual nine."),
} for scope, count in scope_counts.items()]

unmapped_scopes = sorted(
    (cwe_id for cwe_id, scope_names in scopes_by_weakness.items() if not scope_names),
    key=int,
)

print(
    f"{len(weaknesses) - len(unmapped_scopes):,} of {len(weaknesses):,} weaknesses list at least one scope. "
    f"{len(unmapped_scopes):,} list none.\n"
    "Each weakness is counted once per scope it names, so the bars overlap."
)
show_groups(scope_groups, "Weaknesses by consequence scope", "#00A087")

same_as_pillars = set(unmapped_scopes) == set(unmapped_pillars)
note = (
    "These are the same deprecated entries that have no pillar. "
    if same_as_pillars
    else "These entries have an empty Common Consequences list. "
)
display(Markdown(f"**No scope listed ({len(unmapped_scopes)}).** {note}"))
display(unmapped_table(unmapped_scopes))


## One weakness, all three labels

`assignments` has every weakness. The rows below are a few familiar identifiers, so the three labels can be read side by side. Filter `assignments` to look up any other id.


In [ ]:
def joined_categories(cwe_id: str) -> str:
    ids = categories_by_weakness.get(cwe_id, [])
    if not ids:
        return "Unmapped"
    names = sorted(record_name(categories[cid]) for cid in ids)
    return "; ".join(names)


def pillar_label(cwe_id: str) -> str:
    pillar_id = pillar_by_weakness.get(cwe_id)
    if pillar_id is None:
        return "Unmapped"
    return f"{cwe_label(pillar_id)} {record_name(weaknesses[pillar_id])}"


def cwe_sort_key(series: pd.Series) -> pd.Series:
    return series.str.split("-").str[-1].astype(int)


assignments = pd.DataFrame([
    {
        "CWE": cwe_label(cwe_id),
        "Name": record_name(record),
        "Abstraction": record.get("Abstraction", ""),
        "Status": record.get("Status", ""),
        "Pillar": pillar_label(cwe_id),
        "Software Development": joined_categories(cwe_id),
        "Scopes": ", ".join(scopes_by_weakness.get(cwe_id, [])) or "Unmapped",
    }
    for cwe_id, record in weaknesses.items()
]).sort_values("CWE", key=cwe_sort_key, ignore_index=True)

examples = ["CWE-79", "CWE-89", "CWE-119", "CWE-787", "CWE-416", "CWE-798", "CWE-352"]
print(f"assignments has {len(assignments):,} weaknesses.")
with pd.option_context("display.max_colwidth", 80):
    display(assignments.set_index("CWE").loc[examples])


## Same label in all three methods

The three methods do not share a vocabulary. A pillar name, a Software Development category, and a consequence scope are different kinds of labels, so they cannot come out as one identical string.

What can match is the grouping. Two weaknesses are labeled the same no matter which method you use when they have the same pillar, the same Software Development categories, and the same set of consequence scopes. Scope order does not matter. This table keeps only those shared combinations. A weakness whose combination is unique is left out.

In [ ]:
def sorted_scopes(cwe: str) -> str:
    number = cwe.split("-", 1)[1]
    names = scopes_by_weakness.get(number, [])
    return ", ".join(sorted(names)) if names else "Unmapped"


def member_list(frame: pd.DataFrame) -> str:
    ordered = frame.sort_values("CWE", key=cwe_sort_key)
    return "; ".join(f"{cwe} {name}" for cwe, name in zip(ordered["CWE"], ordered["Name"]))


labeled = assignments.copy()
labeled["Scopes"] = labeled["CWE"].map(sorted_scopes)

same_labels = (
    labeled.groupby(["Pillar", "Software Development", "Scopes"], sort=False)
    .apply(
        lambda frame: pd.Series({
            "Weaknesses": len(frame),
            "Members": member_list(frame),
        }),
        include_groups=False,
    )
    .reset_index()
)

same_labels = (
    same_labels.loc[same_labels["Weaknesses"] >= 2]
    .sort_values(
        ["Weaknesses", "Pillar", "Software Development", "Scopes"],
        ascending=[False, True, True, True],
        ignore_index=True,
    )[["Weaknesses", "Pillar", "Software Development", "Scopes", "Members"]]
)

shared = int(same_labels["Weaknesses"].sum())
unique = len(labeled) - shared
print(
    f"{len(same_labels):,} shared label combinations cover {shared:,} weaknesses.\n"
    f"{unique:,} weaknesses have a combination no other weakness shares, so they are not in this table."
)
with pd.option_context("display.max_rows", 200, "display.max_colwidth", None):
    display(same_labels)
